## Atividade 4: Modelagem com Multilayer Perceptron (MLP)

### 1) Dados:

Utilize o conjunto de dados normalizados (com MinMaxScaler), conforme utilizou na tarefa anterior.

### 2) Definição do Modelo MLP:

O modelo MLPRegressor é uma rede neural feedforward de múltiplas camadas, com os seguintes parâmetros principais:

- n_hidden_layers: define o número de camadas ocultas.
- hidden_layers_size: define o número de neurônios em cada camada oculta.
- activation: função de ativação (ReLU, tanh, logistic, etc.).
- learning_rate_init: taxa de aprendizado inicial.
- solver: algoritmo de otimização (Podem testar com 'adam' e 'nadam')
- alpha: regularização L2.

Obs.: Regularização é uma técnica usada para evitar overfitting em modelos de aprendizado de máquina, adicionando uma penalização nos pesos na função de custo. 

### 3) Otimização dos hiperparâmetros:

Use o Optuna (ou Gridsearch) para otimizar os parâmetros do item 2. Use, por exemplo:

- n_hidden_layers: entre 1 e 4.
- hidden_layers_size: entre o número de neurônios de entrada (features) e 32.
- activation: ReLU e tanh.
- learning_rate_init: entre 1e-5 e 1e-2.
- solver:  adam e nadam
- alpha: regularização L2.

A regularização L2 é um hiperparâmetro que controla o “peso” da penalização. Se for muito pequeno, o modelo pode sobreajustar (overfit); se for muito grande, o modelo pode subajustar (underfit). Peça para o Optuna buscar o valor ideal dentro do intervalo 1e-6, 1e-2, por exemplo.

### 4) Treine o modelo final com os melhores hyperparâmetros e compare os resultados com o modelo Random Forest. 

### 1: Import das bibliotecas

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
import optuna
from optuna.samplers import TPESampler
from sklearn.exceptions import ConvergenceWarning
import warnings

warnings.filterwarnings("ignore", category=ConvergenceWarning)


run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"Run ID: {run_id}")

Run ID: 20251115_180040


/home/red/git/jupyter-GEX1090/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 2: Dados

In [2]:
x_train = pd.read_csv('data/X_train_scaled.csv', index_col=0)
y_train = pd.read_csv('data/y_train_scaled.csv', index_col=0)

x_val = pd.read_csv('data/X_val_scaled.csv', index_col=0)
y_val = pd.read_csv('data/y_val_scaled.csv', index_col=0)

x_test = pd.read_csv('data/X_test_scaled.csv', index_col=0)
y_test = pd.read_csv('data/y_test_scaled.csv', index_col=0)

RANDOM_SEED = 27
np.random.seed(RANDOM_SEED)

### 3: Definição do Modelo MLP

In [3]:
mlp = MLPRegressor(
    hidden_layer_sizes=(100, 50),
    activation='relu',
    solver='adam',
    learning_rate='adaptive',
    max_iter=500,
    random_state=RANDOM_SEED
)

mlp.fit(x_train, y_train.values.ravel())

val_predictions = mlp.predict(x_val)

rmse = root_mean_squared_error(y_val, val_predictions)
mae = mean_absolute_error(y_val, val_predictions)
r2 = r2_score(y_val, val_predictions)

print(f'Validação -> RMSE: {rmse:.6f} | MAE: {mae:.6f} | R²: {r2:.6f}')

Validação -> RMSE: 0.013123 | MAE: 0.008494 | R²: 0.997726


### 4: Otimização dos hiperparâmetros

In [4]:
n_features = x_train.shape[1]

def objective(trial):
    # Espaço de busca
    n_hidden_layers = trial.suggest_int("n_hidden_layers", 1, 4)
    hidden_units = trial.suggest_int(
        "hidden_units",
        low=min(n_features, 32),
        high=max(n_features, 32)
    )
    activation = trial.suggest_categorical("activation", ["relu", "tanh"])
    learning_rate_init = trial.suggest_float("learning_rate_init", 1e-5, 1e-2, log=True)
    alpha = trial.suggest_float("alpha", 1e-6, 1e-2, log=True)
    solver = "adam"

    hidden_layer_sizes = tuple([hidden_units] * n_hidden_layers)

    model = MLPRegressor(
        hidden_layer_sizes=hidden_layer_sizes,
        activation=activation,
        solver=solver,
        learning_rate="adaptive",
        learning_rate_init=learning_rate_init,
        alpha=alpha,
        max_iter=1000,
        early_stopping=True,
        n_iter_no_change=20,
        random_state=RANDOM_SEED
    )

    model.fit(x_train, y_train.values.ravel())
    preds_val = model.predict(x_val)

    rmse = root_mean_squared_error(y_val, preds_val)
    return rmse  # Optuna minimiza

study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=RANDOM_SEED))
study.optimize(objective, n_trials=30, show_progress_bar=True, n_jobs=-1)

print("Melhor RMSE (val):", study.best_value)
print("Melhores parâmetros:")
for k, v in study.best_params.items():
    print(f"  - {k}: {v}")
    
trials_df = study.trials_dataframe()
trials_df.to_csv(f"data/mlp_optuna_trials_{run_id}.csv", index=False)

pd.DataFrame([study.best_params]).to_csv(f"data/mlp_best_params_{run_id}.csv", index=False)

[I 2025-11-15 18:00:44,044] A new study created in memory with name: no-name-d04f62de-1b2b-4bad-8dfe-b37153379dbb
Best trial: 11. Best value: 0.00858088:   3%|▎         | 1/30 [07:37<3:40:58, 457.20s/it]

[I 2025-11-15 18:08:21,116] Trial 11 finished with value: 0.008580883550423881 and parameters: {'n_hidden_layers': 2, 'hidden_units': 29, 'activation': 'relu', 'learning_rate_init': 0.009853859727873001, 'alpha': 4.341144247589677e-06}. Best is trial 11 with value: 0.008580883550423881.


Best trial: 11. Best value: 0.00858088:   7%|▋         | 2/30 [09:44<2:02:48, 263.17s/it]

[I 2025-11-15 18:10:28,424] Trial 8 finished with value: 0.01224593701042003 and parameters: {'n_hidden_layers': 2, 'hidden_units': 27, 'activation': 'relu', 'learning_rate_init': 0.0005505096152244978, 'alpha': 0.0006376955644362439}. Best is trial 11 with value: 0.008580883550423881.


Best trial: 7. Best value: 0.00705427:  10%|█         | 3/30 [10:14<1:10:30, 156.68s/it] 

...

[I 2025-11-15 19:06:46,348] Trial 1 finished with value: 0.02875138441753232 and parameters: {'n_hidden_layers': 4, 'hidden_units': 24, 'activation': 'tanh', 'learning_rate_init': 3.0802899816098545e-05, 'alpha': 0.007550243130229034}. Best is trial 0 with value: 0.006216957218340462.


Best trial: 0. Best value: 0.00621696:  97%|█████████▋| 29/30 [1:06:45<01:37, 97.29s/it] 

[I 2025-11-15 19:07:29,753] Trial 25 finished with value: 0.022751867274883833 and parameters: {'n_hidden_layers': 4, 'hidden_units': 32, 'activation': 'tanh', 'learning_rate_init': 3.785758609734037e-05, 'alpha': 9.608109672623026e-06}. Best is trial 0 with value: 0.006216957218340462.


Best trial: 0. Best value: 0.00621696: 100%|██████████| 30/30 [1:07:08<00:00, 134.28s/it]

[I 2025-11-15 19:07:52,309] Trial 5 finished with value: 0.038160201199185444 and parameters: {'n_hidden_layers': 3, 'hidden_units': 31, 'activation': 'tanh', 'learning_rate_init': 1.1011320478489441e-05, 'alpha': 9.450918480313856e-05}. Best is trial 0 with value: 0.006216957218340462.
Melhor RMSE (val): 0.006216957218340462
Melhores parâmetros:
  - n_hidden_layers: 4
  - hidden_units: 30
  - activation: relu
  - learning_rate_init: 0.003456301036069964
  - alpha: 1.5596988905001667e-05


In [5]:
# (Opcional) Treina o modelo final com os melhores hiperparâmetros e avalia em validação e teste

best = study.best_params
best_hidden = tuple([best["hidden_units"]] * best["n_hidden_layers"])

mlp_best = MLPRegressor(
    hidden_layer_sizes=best_hidden,
    activation=best["activation"],
    solver="adam",
    learning_rate="adaptive",
    learning_rate_init=best["learning_rate_init"],
    alpha=best["alpha"],
    max_iter=1000,
    early_stopping=True,
    n_iter_no_change=20,
    random_state=RANDOM_SEED
)

# Treina no conjunto de treino
mlp_best.fit(x_train, y_train.values.ravel())

# Métricas em validação
val_pred = mlp_best.predict(x_val)
val_rmse = root_mean_squared_error(y_val, val_pred)
val_mae = mean_absolute_error(y_val, val_pred)
val_r2 = r2_score(y_val, val_pred)
print(f"Validação -> RMSE: {val_rmse:.6f} | MAE: {val_mae:.6f} | R²: {val_r2:.6f}")

# Re-treina no conjunto treino+validação para obter o modelo final antes do teste
x_tr_full = pd.concat([x_train, x_val], axis=0)
y_tr_full = pd.concat([y_train, y_val], axis=0)

mlp_best.fit(x_tr_full, y_tr_full.values.ravel())

# Métricas em teste
test_pred = mlp_best.predict(x_test)
test_rmse = root_mean_squared_error(y_test, test_pred)
test_mae = mean_absolute_error(y_test, test_pred)
test_r2 = r2_score(y_test, test_pred)
print(f"Teste     -> RMSE: {test_rmse:.6f} | MAE: {test_mae:.6f} | R²: {test_r2:.6f}")

metrics_df = pd.DataFrame(
    [
        {"split": "val", "rmse": float(val_rmse), "mae": float(val_mae), "r2": float(val_r2)},
        {"split": "test", "rmse": float(test_rmse), "mae": float(test_mae), "r2": float(test_r2)},
    ]
)
metrics_df.to_csv(f"data/mlp_final_metrics_{run_id}.csv", index=False)

Validação -> RMSE: 0.006217 | MAE: 0.004335 | R²: 0.999490
Teste     -> RMSE: 0.003746 | MAE: 0.002603 | R²: 0.999816
